<a href="https://www.kaggle.com/code/thisisrick25/imdb-sentiment-linear-svm?scriptVersionId=355204686" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# IMDb Sentiment Classification using Linear SVM

**Objective:** Use the preprocessed IMDb reviews and TF-IDF representations to build a sentiment classifier using `LinearSVC`.

This notebook addresses parts **(a)–(i)** of the assignment.

## Assignment Requirements

- **(a)** Train a Linear SVM using `LinearSVC`.
- **(b)** Experiment with `C ∈ {0.01, 0.1, 1, 10}`.
- **(c)** Report accuracy, precision, recall, and F1-score for every `C`.
- **(d)** Plot F1-score against `C` and identify the best value.
- **(e)** Compare unigram TF-IDF with unigram+bigrams TF-IDF.
- **(f–g)** Identify and display the 10 strongest positive and 10 strongest negative features with coefficient values.
- **Heart:** Discuss whether influential features make intuitive sense.
- **(i)** Find five incorrectly classified reviews containing strong sentiment features and analyze the errors.

In [ ]:
import os
import re
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

print("Libraries imported.")

## 1. Load the IMDb Dataset / Reuse the Preprocessed Data

If the previous preprocessing notebook has already created `df` with `preprocessed_review` and `sentiment`, this notebook reuses those columns.

Otherwise, it searches the Kaggle input directory for `IMDB Dataset.csv` and performs the same preprocessing pipeline.

In [ ]:
# Reuse an existing preprocessed dataframe when available.
if "df" not in globals() or not {"preprocessed_review", "sentiment"}.issubset(df.columns):
    dataset_path = None

    for root, dirs, files in os.walk("/kaggle/input"):
        if "IMDB Dataset.csv" in files:
            dataset_path = os.path.join(root, "IMDB Dataset.csv")
            break

    if dataset_path is None:
        raise FileNotFoundError(
            "IMDB Dataset.csv was not found. Add the IMDb Dataset of 50K Movie Reviews "
            "through Kaggle Notebook -> Add Input."
        )

    print("Dataset:", dataset_path)
    df = pd.read_csv(dataset_path)

    # Recreate the same preprocessing pipeline used for the previous IMDb assignment.
    import nltk
    from nltk.corpus import stopwords
    from nltk.stem import WordNetLemmatizer

    nltk.download("stopwords", quiet=True)
    nltk.download("wordnet", quiet=True)
    nltk.download("omw-1.4", quiet=True)

    stop_words = set(stopwords.words("english"))
    lemmatizer = WordNetLemmatizer()

    def preprocess_review(review):
        text = str(review).lower()
        text = re.sub(r"<[^>]+>", " ", text)
        text = re.sub(r"https?://\S+|www\.\S+", " ", text)
        text = re.sub(r"[^a-z\s]", " ", text)
        tokens = text.split()
        tokens = [word for word in tokens if word not in stop_words]
        tokens = [lemmatizer.lemmatize(word) for word in tokens]
        return " ".join(tokens)

    df["preprocessed_review"] = df["review"].apply(preprocess_review)

else:
    print("Using existing preprocessed dataframe from the previous IMDb preprocessing work.")

print("Dataset shape:", df.shape)
print(df[["review", "preprocessed_review", "sentiment"]].head())

In [ ]:
# Encode sentiment: negative = 0, positive = 1
df = df.dropna(subset=["preprocessed_review", "sentiment"]).copy()

df["sentiment_num"] = (
    df["sentiment"]
    .astype(str)
    .str.lower()
    .map({"negative": 0, "positive": 1})
)

if df["sentiment_num"].isna().any():
    raise ValueError("Unexpected sentiment labels found.")

print(df["sentiment_num"].value_counts().sort_index())

## 2. Train-Test Split

An **80/20 stratified split** is used so that the positive/negative class proportions are preserved in both sets.

In [ ]:
X_text = df["preprocessed_review"]
y = df["sentiment_num"]

# Keep raw reviews aligned with the same indices for later error analysis.
X_raw = df["review"]

X_train_text, X_test_text, y_train, y_test = train_test_split(
    X_text,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

X_train_raw = X_raw.loc[X_train_text.index]
X_test_raw = X_raw.loc[X_test_text.index]

print("Training samples:", len(X_train_text))
print("Testing samples :", len(X_test_text))
print("Training class distribution:")
print(y_train.value_counts().sort_index())
print("Testing class distribution:")
print(y_test.value_counts().sort_index())

## 3. TF-IDF Representations

Two representations are used:

1. **Unigram TF-IDF:** `ngram_range=(1, 1)`
2. **Unigram + bigram TF-IDF:** `ngram_range=(1, 2)`

The vectorizers are fitted **only on the training data** to avoid test-set vocabulary leakage.

In [ ]:
tfidf_uni = TfidfVectorizer(ngram_range=(1, 1))
X_train_uni = tfidf_uni.fit_transform(X_train_text)
X_test_uni = tfidf_uni.transform(X_test_text)

tfidf_bi = TfidfVectorizer(ngram_range=(1, 2))
X_train_bi = tfidf_bi.fit_transform(X_train_text)
X_test_bi = tfidf_bi.transform(X_test_text)

print("Unigram TF-IDF:")
print("  Training shape:", X_train_uni.shape)
print("  Testing shape :", X_test_uni.shape)

print("\nUnigram + bigram TF-IDF:")
print("  Training shape:", X_train_bi.shape)
print("  Testing shape :", X_test_bi.shape)

# (a)–(d) Linear SVM: Effect of the Regularization Parameter C

A `LinearSVC` classifier is trained using unigram TF-IDF features for:

- `C = 0.01`
- `C = 0.1`
- `C = 1`
- `C = 10`

The test-set accuracy, precision, recall, and F1-score are reported for each value.

In [ ]:
C_values = [0.01, 0.1, 1, 10]
c_results = []
c_models = {}

for C in C_values:
    model = LinearSVC(C=C, random_state=RANDOM_STATE, max_iter=5000)
    model.fit(X_train_uni, y_train)

    y_pred = model.predict(X_test_uni)

    c_results.append({
        "C": C,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1-score": f1_score(y_test, y_pred, zero_division=0)
    })
    c_models[C] = model

c_results_df = pd.DataFrame(c_results)
c_results_df

In [ ]:
# Identify the C with the best test F1-score.
best_C = c_results_df.loc[c_results_df["F1-score"].idxmax(), "C"]

print("Best C based on test F1-score:", best_C)

In [ ]:
# (d) Plot F1-score against C.
plt.figure(figsize=(7, 5))
plt.plot(c_results_df["C"], c_results_df["F1-score"], marker="o")
plt.xscale("log")
plt.xlabel("C (log scale)")
plt.ylabel("F1-score")
plt.title("Linear SVM F1-score vs Regularization Parameter C")
plt.grid(True, alpha=0.3)
plt.show()

### Interpretation of C

The best value of `C` is selected using the highest **test-set F1-score**, since F1 balances precision and recall.

A larger `C` places a stronger penalty on training errors, while a smaller `C` applies stronger regularization.

# (e) Unigram vs Unigram+Bigrams

Using the best `C` identified above, train two Linear SVM models:

1. Unigram TF-IDF
2. Unigram + bigram TF-IDF

Their test-set performance is compared using accuracy, precision, recall, and F1-score.

In [ ]:
best_svm_uni = LinearSVC(C=best_C, random_state=RANDOM_STATE, max_iter=5000)
best_svm_uni.fit(X_train_uni, y_train)
pred_uni = best_svm_uni.predict(X_test_uni)

best_svm_bi = LinearSVC(C=best_C, random_state=RANDOM_STATE, max_iter=5000)
best_svm_bi.fit(X_train_bi, y_train)
pred_bi = best_svm_bi.predict(X_test_bi)

comparison = pd.DataFrame([
    {
        "Representation": "Unigram TF-IDF",
        "Accuracy": accuracy_score(y_test, pred_uni),
        "Precision": precision_score(y_test, pred_uni, zero_division=0),
        "Recall": recall_score(y_test, pred_uni, zero_division=0),
        "F1-score": f1_score(y_test, pred_uni, zero_division=0)
    },
    {
        "Representation": "Unigram + Bigram TF-IDF",
        "Accuracy": accuracy_score(y_test, pred_bi),
        "Precision": precision_score(y_test, pred_bi, zero_division=0),
        "Recall": recall_score(y_test, pred_bi, zero_division=0),
        "F1-score": f1_score(y_test, pred_bi, zero_division=0)
    }
])

comparison

In [ ]:
uni_f1 = comparison.loc[
    comparison["Representation"] == "Unigram TF-IDF", "F1-score"
].iloc[0]

bi_f1 = comparison.loc[
    comparison["Representation"] == "Unigram + Bigram TF-IDF", "F1-score"
].iloc[0]

print(f"F1-score difference (bigram - unigram): {bi_f1 - uni_f1:.4f}")

if bi_f1 > uni_f1:
    print("Adding bigrams improves sentiment classification based on F1-score.")
elif bi_f1 < uni_f1:
    print("Adding bigrams does not improve sentiment classification based on F1-score.")
else:
    print("Adding bigrams gives the same F1-score as unigrams.")

# (f)–(g) Most Influential Positive and Negative Features

For a linear SVM, the learned coefficient associated with each TF-IDF feature indicates its influence on the decision function:

- **Large positive coefficient:** pushes the prediction toward positive sentiment.
- **Large negative coefficient:** pushes the prediction toward negative sentiment.

The feature analysis below uses the **better-performing representation** from the unigram vs bigram comparison.

In [ ]:
# Select the better representation using F1-score.
if bi_f1 > uni_f1:
    selected_model = best_svm_bi
    selected_vectorizer = tfidf_bi
    selected_X_test = X_test_bi
    selected_pred = pred_bi
    selected_representation = "Unigram + Bigram TF-IDF"
else:
    selected_model = best_svm_uni
    selected_vectorizer = tfidf_uni
    selected_X_test = X_test_uni
    selected_pred = pred_uni
    selected_representation = "Unigram TF-IDF"

print("Selected representation for feature analysis:", selected_representation)

In [ ]:
feature_names = np.array(selected_vectorizer.get_feature_names_out())
coefficients = selected_model.coef_.ravel()

# Ten strongest positive features = largest positive coefficients.
positive_indices = np.argsort(coefficients)[-10:][::-1]

# Ten strongest negative features = most negative coefficients.
negative_indices = np.argsort(coefficients)[:10]

positive_features = pd.DataFrame({
    "Feature": feature_names[positive_indices],
    "Coefficient": coefficients[positive_indices]
})

negative_features = pd.DataFrame({
    "Feature": feature_names[negative_indices],
    "Coefficient": coefficients[negative_indices]
})

print("Top 10 positive features:")
display(positive_features)

print("\nTop 10 negative features:")
display(negative_features)

## Heart — Do the Influential Features Make Intuitive Sense?

The strongest positive and negative features should generally correspond to sentiment-bearing language.

Typical examples might include strongly positive terms such as *excellent*, *amazing*, or *wonderful*, and strongly negative terms such as *worst*, *awful*, or *boring*.

However, the exact features depend on the training data and preprocessing. Bigram features can also capture more specific phrases such as combinations of words whose sentiment is clearer than either word alone.

The tables above should therefore be inspected to determine whether the learned coefficients agree with intuitive sentiment.

# (i) Error Analysis

We now identify **five incorrectly classified test reviews** that contain at least one feature among the strongest positive or negative SVM features.

For each review, we display:

- the original review,
- actual sentiment,
- predicted sentiment,
- the relevant strong feature(s),
- their SVM coefficient values,
- and the feature contribution to the SVM decision.

This helps explain why a review containing strongly sentiment-associated words may still be classified incorrectly.

In [ ]:
# Strong features used for error analysis.
strong_feature_indices = np.concatenate([positive_indices, negative_indices])
strong_feature_set = set(feature_names[strong_feature_indices])

# Map each feature to its learned coefficient.
coef_map = dict(zip(feature_names, coefficients))

# Use the selected model's decision scores.
decision_scores = selected_model.decision_function(selected_X_test)

error_rows = []

for row_position, (idx, true_label, pred_label, score) in enumerate(
    zip(X_test_text.index, y_test.values, selected_pred, decision_scores)
):
    if true_label == pred_label:
        continue

    # Get non-zero TF-IDF features for this review.
    row = selected_X_test[row_position]
    feature_indices = row.indices

    relevant = []
    for feature_idx in feature_indices:
        feature = feature_names[feature_idx]
        if feature in strong_feature_set:
            tfidf_value = row[0, feature_idx]
            coefficient = coef_map[feature]
            contribution = tfidf_value * coefficient
            relevant.append((feature, coefficient, tfidf_value, contribution))

    if relevant:
        # Rank relevant features by absolute contribution.
        relevant = sorted(relevant, key=lambda x: abs(x[3]), reverse=True)

        error_rows.append({
            "index": idx,
            "actual": "positive" if true_label == 1 else "negative",
            "predicted": "positive" if pred_label == 1 else "negative",
            "decision_score": score,
            "strong_features": ", ".join(x[0] for x in relevant[:5]),
            "feature_coefficients": ", ".join(
                f"{x[0]}={x[1]:.4f}" for x in relevant[:5]
            ),
            "feature_contributions": ", ".join(
                f"{x[0]}={x[3]:.4f}" for x in relevant[:5]
            ),
            "review": X_test_raw.loc[idx]
        })

    if len(error_rows) >= 5:
        break

error_df = pd.DataFrame(error_rows)

if len(error_df) < 5:
    print(
        f"Only {len(error_df)} qualifying errors were found. "
        "This can happen depending on the train/test split and learned features."
    )
else:
    display(error_df[[
        "index",
        "actual",
        "predicted",
        "decision_score",
        "strong_features",
        "feature_coefficients",
        "feature_contributions",
        "review"
    ]])

## Error Analysis Discussion

For each of the five errors above, examine the relationship between the review text and the influential features.

Possible explanations include:

- **Negation:** a strongly positive word may appear in a negative context, e.g. “not excellent”.
- **Mixed sentiment:** the review may contain both positive and negative opinions.
- **Contrast:** constructions such as “although”, “but”, or “however” can reverse the importance of individual words.
- **Context dependence:** a word that is usually positive or negative may have a different meaning in a particular sentence.
- **Bag-of-words limitation:** TF-IDF represents features largely independently and does not fully understand sentence-level meaning.
- **Bigram limitations:** even with bigrams, the model still uses a linear decision boundary and may not capture longer-range linguistic relationships.

The `feature_contributions` column shows how the TF-IDF value multiplied by the learned coefficient contributes to the SVM decision for the selected review.

# Final Summary

This notebook completed the requested Linear SVM sentiment-classification experiment:

- Tested `C = {0.01, 0.1, 1, 10}`.
- Reported accuracy, precision, recall, and F1-score.
- Plotted F1-score against `C`.
- Identified the best `C`.
- Compared unigram TF-IDF against unigram+bigrams TF-IDF.
- Identified the 10 strongest positive and 10 strongest negative features.
- Reported their SVM coefficient values.
- Investigated whether the learned features make intuitive sentiment sense.
- Performed error analysis on five incorrectly classified reviews containing strong sentiment features.